__Homework 6__

The goal of this homework is to finetune a pretrained LLM to be better at mathematical reasoning.

As before, we will use a Llama-3-8B model. We will finetune the model on a small subset of school-math instructions. We will use LoRA for efficient finetuning. We will notice how the model's ability to reason in simple math questions is significantly improved after finetuning.

Most of the code is given to you. Your task is to implement the optimizer and tune its hyperparameters properly. One epoch of finetuning is sufficient for this homework


In [2]:
import os
import subprocess
import sys

EXPECTED = os.path.join("Homework 6", ".venv-hw6", "Scripts", "python.exe")

print("Python executable:", sys.executable)
print("Expected kernel:   ...", EXPECTED)

import torch

print("\nPyTorch version:              ", torch.__version__)
print("CUDA compiled into PyTorch:   ", torch.backends.cuda.is_built())
print("torch.cuda.is_available():    ", torch.cuda.is_available())
print("torch.cuda.device_count():    ", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}:", torch.cuda.get_device_name(i))
else:
    print("  PyTorch sees no CUDA device")

try:
    print("torch.accelerator.current_accelerator():", torch.accelerator.current_accelerator())
except Exception as exc:
    print("torch.accelerator.current_accelerator():", exc)

try:
    smi = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, check=False)
    print("\nnvidia-smi -L:\n", (smi.stdout or smi.stderr).strip() or "(no output)")
except FileNotFoundError:
    print("\nnvidia-smi not found on PATH")

if ".venv-hw6-test" in sys.executable.replace("\\", "/"):
    raise RuntimeError(
        "Wrong Jupyter kernel: you are on .venv-hw6-test (CPU PyTorch). "
        "In VS Code use Kernel -> Select Kernel -> .venv-hw6, then Restart Kernel."
    )

if not torch.backends.cuda.is_built() or not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch in this kernel has no CUDA. Stop the kernel and run: python setup_hw6_env.py --recreate "
        "Then select the .venv-hw6 kernel."
    )

print("\nGPU check passed.")

Python executable: c:\Users\Phili\OneDrive\Documents\Studium\Master\6.Semester\FoF\Exercises\Homework 6\.venv-hw6\Scripts\python.exe
Expected kernel:   ... Homework 6\.venv-hw6\Scripts\python.exe

PyTorch version:               2.6.0+cu124
CUDA compiled into PyTorch:    True
torch.cuda.is_available():     True
torch.cuda.device_count():     1
  GPU 0: NVIDIA GeForce RTX 3090 Ti
torch.accelerator.current_accelerator(): cuda

nvidia-smi -L:
 GPU 0: NVIDIA GeForce RTX 3090 Ti (UUID: GPU-ced1ee2a-7067-1794-fb8f-74128835ac3e)

GPU check passed.


In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [4]:
import os
import ssl


os.environ.setdefault("UNSLOTH_COMPILE_DISABLE", "1")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
ssl.create_default_context()

import datasets
from datasets import load_dataset
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available in this kernel. Run the GPU check cell first and switch to .venv-hw6.")

from unsloth import FastLanguageModel
from transformers import TextStreamer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.



# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/llama-3-8b-bnb-4bit", # [NEW] 15 Trillion token Llama-3
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3090 Ti. Num GPUs = 1. Max memory: 23.988 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128255)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSN

In [6]:
# initialize LoRA parameters and associate them with the model
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj","up_proj", "down_proj",],
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # reducing memory
    random_state = 1,
)

Unsloth 2026.5.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [7]:
# load OpenAI's grade-school-math dataset from the paper Cobbe et al, 'Training Verifiers to Solve Math World Problems', 2021
dataset = load_dataset("qwedsacf/grade-school-math-instructions", split = "train")
print('The dataset has ', len(dataset), '  many entries')

dataset = dataset.select(range(500)) #Take only 500 examples
print('We take ' , len(dataset), ' many entries for this homework ')

print('This is how an entry of the dataset looks like: ' , dataset[0])

README.md:   0%|          | 0.00/852 [00:00<?, ?B/s]

data/train-00000-of-00001-3f5d4168106415(…):   0%|          | 0.00/2.55M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8792 [00:00<?, ? examples/s]

The dataset has  8792   many entries
We take  500  many entries for this homework 
This is how an entry of the dataset looks like:  {'INSTRUCTION': 'This math problem has got me stumped: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nCan you show me the way?', 'RESPONSE': 'Natalia sold 48/2 = 24 clips in May.\nNatalia sold 48+24 = 72 clips altogether in April and May.', 'SOURCE': 'grade-school-math'}


In [8]:
# Prepare the data for finetuning, we concatenate the INSTRUCTION and RESPONSE fields from the grade-school-math instructions dataset
# into one string.

prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts(examples):
    instructions = examples["INSTRUCTION"]
    responses     = examples["RESPONSE"]
    texts = []
    for instruction, response in zip(instructions, responses):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = prompt.format(instruction, response) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts, batched = True)
print(dataset[0]['text'])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
This math problem has got me stumped: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Can you show me the way?

### Response:
Natalia sold 48/2 = 24 clips in May.
Natalia sold 48+24 = 72 clips altogether in April and May.<|end_of_text|>


In [9]:
#Here we check how the pretrained model responds to 3 simple questions before finetuning.
Q1 = "I have 10 apples, my brother took half of them from me, I lost 1, and my friend gave me 3. How many do I have now?"
Q2 = "I earn five euros per hour. I worked two hours yesterday and five hours today. How much did I earn in total?"
Q3 = "In year 2000 I was 20 years old. My sister is 5 years younger than me. How old is she in 2020?"

input1 = tokenizer([prompt.format(Q1, "",)],return_tensors = "pt").to("cuda")
input2 = tokenizer([prompt.format(Q2, "",)],return_tensors = "pt").to("cuda")
input3 = tokenizer([prompt.format(Q3, "",)],return_tensors = "pt").to("cuda")

In [10]:
#Response to first question
_= model.generate(**input1, streamer = TextStreamer(tokenizer), max_new_tokens=70, max_length=None, do_sample=False)

<|begin_of_text|>Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
I have 10 apples, my brother took half of them from me, I lost 1, and my friend gave me 3. How many do I have now?

### Response:
I have 10 apples, my brother took half of them from me, I lost 1, and my friend gave me 3. How many do I have now?

### Explanation:
I have 10 apples, my brother took half of them from me, I lost 1, and my friend gave me 3. How many do I


In [11]:
#Response to second question
_= model.generate(**input2, streamer = TextStreamer(tokenizer), max_new_tokens=70, max_length=None, do_sample=False)

<|begin_of_text|>Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
I earn five euros per hour. I worked two hours yesterday and five hours today. How much did I earn in total?

### Response:
I earned 10 euros in total.<|end_of_text|>


In [13]:
#Response to third question
_= model.generate(**input3, streamer = TextStreamer(tokenizer), max_new_tokens=70, max_length=None, do_sample=False)

<|begin_of_text|>Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
In year 2000 I was 20 years old. My sister is 5 years younger than me. How old is she in 2020?

### Response:
In year 2000 I was 20 years old. My sister is 5 years younger than me. How old is she in 2020?

### Explanation:
In year 2000 I was 20 years old. My sister is 5 years younger than me. How old is she in 2020?

### Instruction:
In year


Implement and run the finetuning here. We recommend using the SFTTrainer (Supervised FineTuning Trainer). One epoch of finetuning is usually enough for the model to answer the questions.

In [16]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

try:
    FastLanguageModel.for_training(model)
except AttributeError:
    model.train()
    for module in model.modules():
        if hasattr(module, "gradient_checkpointing"):
            module.gradient_checkpointing = True

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=1,
        output_dir="outputs",
    ),
)

trainer_stats = trainer.train()

try:
    FastLanguageModel.for_inference(model)
except AttributeError:
    model.eval()

Unsloth: Tokenizing ["text"]:   0%|          | 0/500 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 63
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,971,520 of 8,051,232,768 (0.26% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.748500
20,1.198500
30,1.119300
40,1.067900
50,0.993100
60,1.010300


In [17]:
#Response to first question after finetuning
_= model.generate(**input1, streamer = TextStreamer(tokenizer), max_new_tokens=70, max_length=None, do_sample=False)

<|begin_of_text|>Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
I have 10 apples, my brother took half of them from me, I lost 1, and my friend gave me 3. How many do I have now?

### Response:
I have 10-5=5 apples
I have 5-1=4 apples
I have 4+3=7 apples<|end_of_text|>


In [18]:
#Response to second question after finetuning
_= model.generate(**input2, streamer = TextStreamer(tokenizer), max_new_tokens=70, max_length=None, do_sample=False)

<|begin_of_text|>Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
I earn five euros per hour. I worked two hours yesterday and five hours today. How much did I earn in total?

### Response:
I earned 5 euros * 2 hours = 10 euros yesterday.
I earned 5 euros * 5 hours = 25 euros today.
I earned 10 euros + 25 euros = 35 euros in total.<|end_of_text|>


In [19]:
#Response to third question after finetuning
_= model.generate(**input3, streamer=TextStreamer(tokenizer), max_new_tokens=70, max_length=None, do_sample=False)

<|begin_of_text|>Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
In year 2000 I was 20 years old. My sister is 5 years younger than me. How old is she in 2020?

### Response:
In 2020 I will be 20+20=40 years old.
My sister will be 5+20=25 years old in 2020.<|end_of_text|>
